# Citi, *Sell Blues convexity adjustments, hedged* — backtested in its own framework

**Sources.** Two Citi notes by Ruslan Bikbov / Jason Williams, the first
citing the second:

* **`print (12).pdf`** — North America Rates Trade Idea, **09 February 2017**,
  *"Sell Blues convexity adjustments, hedged"*. The ticket and Figures 1-6.
* **`print (15/18).pdf`** — US Rates Weekly, **13 January 2017**, *"Swearing in
  huge expectations"*, §*Smart convexity sells*. Figures 16-22, including the
  pack-by-pack screen this backtest selects from.

The note's own words:

> We sell \$200k DV01 of Blues convexity adjustments, i.e. buy 2000 of H0-Z0
> Eurodollar packs (2000 of each of the four contracts) and pay \$2bn on a
> matched-maturity (3/18/20-3/17/21) CME cleared swap at 8.8bp of spread. …
> We hedge the trade by paying the belly of the 2s5s10s swap fly with notional
> weights of \$147mm/-\$85.6mm/\$20.89mm (0.705/-1/0.465 DV01 weights) at
> -18.2bp in terms of the level of the DV01-weighted fly. … We set the target
> at +\$600k profit with the stop at -\$350k loss.

> To build a more optimal hedging strategy, we regressed Blues CA on 2y, 5y
> and 10y swap rates … with the fitted value effectively being a 2s5s10s fly
> with 0.705/-1/0.465 DV01 weights (Figure 6). The CA is about 3bp (about 2
> sigmas) wide to the fly … Our trade is constructed as a convergence trade
> between the Blues CA and the 2s5s10s fly, precisely as illustrated in
> Figure 6.

## What this notebook is

A **pre-registered backtest of that rule**, as a rule: the screen across the
strip, the fitted hedge, the five-condition entry conjunction, the dollar
target and stop, short only, held through the quarterly rolls on dated
instruments. The declared cell list is frozen in
`docs/convexityrv/citi-framework-preregistration.md`, committed before any
P&L was computed, and the test suite counts the code's cells against that
document.

It is **not** block 4 again. Block 4 (`gv_grid`, PR #496) scored 298 cells of
"CA versus a butterfly, z-score in, z-score out" and found nothing; that grid
used a rolling univariate beta against a fixed 50/50 fly, froze it at entry,
picked one structure per cell, entered on `|z| ≥ 2`, exited on z, traded both
sides, and sat flat across every roll. Citi's framework does none of those
things. The pre-registration's §0 table is the line-by-line difference.

**The standing caveat.** This is the fifth pass over the same CA panel
(`strat2`, `cavf`/PR #492, `gv`/PR #496, the reproduction/PR #499, now this).
Declaring one rule today does not undo four prior searches, and every number
below carries that.

**Everything here reads prebuilt artifacts.** The panel, the grid, the engine
certification and the control battery are produced by the `_p4_*.py` runners;
this notebook loads them and does no market-data I/O of its own.

In [1]:
import os
import sys
import warnings

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")
sys.path.append("../../..")
warnings.filterwarnings("ignore")

import json
import math
import pathlib
from dataclasses import dataclass

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

from BT.trade_dashboard import compare_curves, trade_dashboard
from RVUtils.ConvexityRV import citi_engine as CE
from RVUtils.ConvexityRV import citi_fv as FV
from RVUtils.ConvexityRV import citi_rule as R
from RVUtils.ConvexityRV import citi_screen as SC

print(f"pandas {pd.__version__}   numpy {np.__version__}")

pandas 2.3.1   numpy 2.2.6


## CONFIG

Every knob, and the note's own printed value beside it. These are the
**declared** values of the pre-registration; the notebook does not re-choose
any of them.

In [2]:
@dataclass(frozen=True)
class Config:
    data_dir: str = "../../data/convexity_rv"
    prereg: str = "../../../docs/convexityrv/citi-framework-preregistration.md"

    # ---- the note's printed ticket, verbatim ----------------------------
    #: "We sell $200k DV01 of Blues convexity adjustments"
    ca_dv01: float = 200_000.0
    #: "at 8.8bp of spread" / "at -18.2bp in terms of the level of the fly"
    entry_ca_bp: float = 8.8
    entry_fly_bp: float = -18.2
    #: Figure 6: CA_bp = 9.7 + 20.6 * (-0.705*r2 + r5 - 0.465*r10), rates PERCENT
    fig6_a: float = FV.CITI_FEB2017_A
    fig6_b: float = FV.CITI_FEB2017_B
    fig6_w2: float = FV.CITI_FEB2017_W2
    fig6_w10: float = FV.CITI_FEB2017_W10
    #: "target at +$600k profit with the stop at -$350k loss"
    target_usd: float = 600_000.0
    stop_usd: float = -350_000.0
    #: the note's own exit: closed 6-Jun-2017 at 6.6bp CA / -16.5bp fly, and
    #: Citi's table records +$552k against a +$600k target
    exit_ca_bp: float = 6.6
    exit_fly_bp: float = -16.5
    recorded_pnl_usd: float = 552_000.0
    #: "The CA is about 3bp (about 2 sigmas) wide to the fly"
    fig6_wide_bp: float = 3.0
    #: Figure 20's H0-Z0 row, close of 12-Jan-2017 (Eurodollars)
    fig20_ca_bp: float = 10.02
    fig20_vs_model_bp: float = 4.61
    fig20_roll_bp: float = 1.30
    fig20_implied: float = 125.5
    fig20_realized: float = 95.1

    # ---- the declared rule ----------------------------------------------
    #: the two rungs of the declared threshold ladder (pre-reg §1 M6)
    z_levels: tuple = R.Z_LEVELS
    #: Figure 20 prints Impl/Rlzd 1.3 on the row Citi traded
    impl_rlzd_min: float = 1.3
    #: "historically high" dealer longs
    z_pos_min: float = 1.0
    #: the reproduction's declared fair-value refit window
    fit_window_bd: int = 504
    #: Citi's two published trades ran 82 and 45 business days
    max_hold_bd: int = 126
    #: fills lag decisions by one mark
    exec_lag_bd: int = 1
    #: 23 declared cells (pre-reg §7)
    n_declared: int = 23


CFG = Config()
DATA = pathlib.Path(CFG.data_dir)

P = pd.read_parquet(DATA / "p4_citi.parquet")
P.index = pd.to_datetime(P.index)
PRE = json.loads((DATA / "p4_preflight.json").read_text())
PROBE = json.loads((DATA / "p4_conjunction_probe.json").read_text())
TIE = json.loads((DATA / "p4_tieout_repro.json").read_text())
CERTIN = json.loads((DATA / "p4_input_certification.json").read_text())
STATS = pd.read_parquet(DATA / "p4_grid_stats.parquet")
EPS = pd.read_parquet(DATA / "p4_grid_episodes.parquet")
RET = pd.read_parquet(DATA / "p4_grid_returns.parquet")
BIND = pd.read_parquet(DATA / "p4_grid_binding.parquet")
GMETA = json.loads((DATA / "p4_grid_meta.json").read_text())
ENG = pd.read_parquet(DATA / "p4_engine_stats.parquet")
EQ = pd.read_parquet(DATA / "p4_engine_equity.parquet")
CTRL = json.loads((DATA / "p4_controls.json").read_text())

SPAN = float(PRE["span_tradeable_years"])
BAR = float(GMETA["bar_annualised_23"])
print(f"panel {P.shape}  {P.index.min().date()}..{P.index.max().date()}")
print(f"tradeable span {SPAN:.3f} y from {PRE['first_defined']}")
print(f"{len(STATS)} declared cells; the bar is E[max SR | null] = {BAR:.4f} "
      f"annualised at {CFG.n_declared} trials")
assert len(STATS) == CFG.n_declared == len(R.declared_cells())

panel (1409, 82)  2021-01-04..2026-08-21
tradeable span 4.682 y from 2021-12-15
23 declared cells; the bar is E[max SR | null] = 0.9066 annualised at 23 trials


## Sign probe

The conventions this backtest can silently get wrong, asserted against live
objects rather than argued about in prose.

In [3]:
_E = __import__("datetime").date(2024, 3, 5)
_X = __import__("datetime").date(2024, 6, 5)
_sp = CE.spec_from_episode(structure="BLUES", side=-1, entry=_E, exit=_X,
                           ca_dv01=CFG.ca_dv01,
                           hedge_path=[(_E, CFG.fig6_b / 100.0, CFG.fig6_w2,
                                        CFG.fig6_w10)])
_q = CE.build_queries(_sp)

# 1. SELLING the CA is BUYING the futures pack and PAYING the matched swap.
_futs = [x for x in _q[_sp.tag] if type(x).__name__ == "STIRFutureQuery"]
_swap = [x for x in _q[_sp.tag] if type(x).__name__ == "IRSwapQuery"][0]
assert len(_futs) == 4
for _f in _futs:
    _k = _f.structure_kwargs
    assert _k["contracts"] > 0 and _k["risk_weights"] == [1.0]
assert _swap.structure_kwargs["bpv"] == CFG.ca_dv01
print(f"side -1 = SELL the CA:  buy {_futs[0].structure_kwargs['contracts']} of "
      f"each of 4 SR3 contracts (risk weight +1), and PAY "
      f"${_swap.structure_kwargs['bpv']:,.0f}/bp on the matched "
      f"{_sp.swap_start}..{_sp.swap_end} swap")

# 2. The fitted fly goes in at ONE times the quoted DV01, not two. gv_engine
#    quotes 2b-f-k with 50/50 wings and must send 2x; this fly is quoted
#    r5 - w2*r2 - w10*r10 and a [w2, 1, w10] package earns exactly bpv per bp.
_fly = _q[_sp.fly_tag(0)][0].structure_kwargs
assert _fly["risk_weights"] == [CFG.fig6_w2, 1.0, CFG.fig6_w10]
assert abs(_fly["bpv"] - (CFG.fig6_b / 100.0) * CFG.ca_dv01) < 1e-6
print(f"hedge: risk weights {_fly['risk_weights']}, belly bpv "
      f"${_fly['bpv']:,.0f}/bp  (= beta {CFG.fig6_b / 100:.3f} x "
      f"${CFG.ca_dv01:,.0f}) -- POSITIVE is PAY the belly, as the note says")

# 3. A negative fitted scale FLIPS the hedge; it does not shrink it.
_neg = CE.spec_from_episode(structure="BLUES", side=-1, entry=_E, exit=_X,
                            ca_dv01=CFG.ca_dv01,
                            hedge_path=[(_E, -CFG.fig6_b / 100.0, 0.5, 0.5)])
assert (_neg.fly_segments[0].leg_dv01_signed
        * _sp.fly_segments[0].leg_dv01_signed) < 0
print(f"a negative b flips the leg: {_sp.fly_segments[0].leg_dv01_signed:+,.0f} "
      f"-> {_neg.fly_segments[0].leg_dv01_signed:+,.0f} per bp")

# 4. The dollar target and stop ARE the note's printed numbers.
_cfg = R.RuleConfig()
assert _cfg.target_usd == CFG.target_usd and _cfg.stop_usd == CFG.stop_usd
assert _cfg.side == -1
print(f"target {_cfg.target_bp:+.2f}bp = ${_cfg.target_usd:+,.0f}   "
      f"stop {_cfg.stop_bp:+.2f}bp = ${_cfg.stop_usd:+,.0f}   "
      f"on ${_cfg.ca_dv01:,.0f} DV01, short only")

# 5. The traded object is the SPREAD, and its beta is bp per bp.
_fit = FV.FairValueFit(CFG.fig6_a, CFG.fig6_b, CFG.fig6_w2, CFG.fig6_w10, 0.9, 500)
assert abs(_fit.beta_bp_per_bp - 0.206) < 1e-9
print(f"beta {CFG.fig6_b} bp of CA per PERCENT of fly = "
      f"{_fit.beta_bp_per_bp:.4f} bp per bp; the fitted weights sum to "
      f"{_fit.w2 + _fit.w10:.3f}, so Citi's published hedge carries "
      f"{_fit.net_weight:+.3f} of outright duration per unit of belly DV01")

side -1 = SELL the CA:  buy 2000 of each of 4 SR3 contracts (risk weight +1), and PAY $200,000/bp on the matched 2027-03-17..2028-03-15 swap
hedge: risk weights [0.705, 1.0, 0.465], belly bpv $41,200/bp  (= beta 0.206 x $200,000) -- POSITIVE is PAY the belly, as the note says
a negative b flips the leg: +41,200 -> -41,200 per bp
target +3.00bp = $+600,000   stop -1.75bp = $-350,000   on $200,000 DV01, short only
beta 20.6 bp of CA per PERCENT of fly = 0.2060 bp per bp; the fitted weights sum to 1.170, so Citi's published hedge carries -0.170 of outright duration per unit of belly DV01


## Known-answer tie-out

Four numbers the note prints must be mutually consistent under this reading of
its conventions, and the one trade whose outcome Citi published must fall out
of the identity this backtest books P&L on. If they do not, the reading is
wrong and everything downstream is fitted to a fiction.

In [4]:
_A2, _A5, _A10 = (FV.annuity(2.0, y) for y in (2, 5, 10))

# (a) the printed NOTIONALS reproduce the printed DV01 WEIGHTS
_w2i = (147.0 * _A2) / (85.6 * _A5)
_w10i = (20.89 * _A10) / (85.6 * _A5)
print(f"(a) DV01 weights implied by $147mm/-$85.6mm/$20.89mm at a flat 2% curve: "
      f"2y {_w2i:.4f} vs {CFG.fig6_w2} ({100 * (_w2i / CFG.fig6_w2 - 1):+.2f}%), "
      f"10y {_w10i:.4f} vs {CFG.fig6_w10} ({100 * (_w10i / CFG.fig6_w10 - 1):+.2f}%)")
assert abs(_w2i / CFG.fig6_w2 - 1) < 0.02 and abs(_w10i / CFG.fig6_w10 - 1) < 0.02

# (b) the regression BETA is the hedge ratio
_belly = _fit.beta_bp_per_bp * CFG.ca_dv01
_belly_n = 85.6 * _A5 * 100.0
print(f"(b) belly DV01 from the Figure-6 beta ${_belly:,.0f}/bp vs from the "
      f"printed notional ${_belly_n:,.0f}/bp ({100 * (_belly / _belly_n - 1):+.1f}%)")
assert abs(_belly / _belly_n - 1) < 0.06

# (c) the printed CARRY is the two printed roll numbers on those DV01s
_carry = 1.3 * CFG.ca_dv01 + 3.2 * _belly
print(f"(c) carry = 1.3bp x ${CFG.ca_dv01:,.0f} + 3.2bp x ${_belly:,.0f} = "
      f"${_carry:,.0f} vs the note's $380,000 "
      f"({100 * (_carry / 380_000 - 1):+.1f}%)")
assert abs(_carry / 380_000 - 1) < 0.08

# (d) the printed RICHNESS falls out of the printed fit
_fitted_at_entry = CFG.fig6_a + CFG.fig6_b * (CFG.entry_fly_bp / 100.0)
_rich = CFG.entry_ca_bp - _fitted_at_entry
print(f"(d) fly {CFG.entry_fly_bp:+.1f}bp -> fitted CA {_fitted_at_entry:.3f}bp, "
      f"CA {CFG.entry_ca_bp}bp -> rich {_rich:+.3f}bp vs 'about "
      f"{CFG.fig6_wide_bp:.0f}bp'")
assert abs(_rich - CFG.fig6_wide_bp) < 0.25

# (e) THE PUBLISHED TRADE'S OWN P&L, from the identity this backtest books on
_pnl = -1 * ((CFG.exit_ca_bp - CFG.entry_ca_bp)
             - _fit.beta_bp_per_bp * (CFG.exit_fly_bp - CFG.entry_fly_bp)) \
       * CFG.ca_dv01
print(f"(e) side*(dCA - beta*dfly)*DV01 = -1 * (("
      f"{CFG.exit_ca_bp}-{CFG.entry_ca_bp}) - {_fit.beta_bp_per_bp:.3f}*("
      f"{CFG.exit_fly_bp}-({CFG.entry_fly_bp}))) * {CFG.ca_dv01:,.0f} = "
      f"${_pnl:,.0f}")
print(f"    Citi's own table records ${CFG.recorded_pnl_usd:,.0f} and the alert "
      f"says 'net +$500k' ({100 * (_pnl / CFG.recorded_pnl_usd - 1):+.1f}% vs "
      "the table)")
assert 4.7e5 < _pnl < 5.6e5

# (f) the promoted fair-value module reproduces the reproduction's own path
print(f"\n(f) citi_fv against the reproduction it was lifted from "
      f"({TIE['window'][0]}..{TIE['window'][1]}, {TIE['fit_window_bd']} bd):")
print(f"    b sign flips {TIE['refit']['b_sign_flips']} at the "
      f"{TIE['refit']['flip_at']} refit;  w2 {TIE['refit']['w2_first']:.2f} -> "
      f"{TIE['refit']['w2_last']:.2f};  b {TIE['refit']['b_first']:+.1f} -> "
      f"{TIE['refit']['b_last']:+.1f}")
print(f"    OOS residual sd: IMM refit {TIE['residuals']['sd_refit']:.3f} bp, "
      f"Citi's fixed 2017 weights {TIE['residuals']['sd_citi']:.3f} bp")
print(f"    fly starts: best {TIE['fly_starts']['best']} at "
      f"{TIE['fly_starts']['best_sd']:.3f} bp; the matched-expiry IMM_13 ranks "
      f"{TIE['fly_starts']['matched_rank']} of {TIE['fly_starts']['n_starts']}")
assert TIE["refit"]["b_sign_flips"] == 1
assert abs(TIE["residuals"]["sd_refit"] - 2.232) < 0.005

# (g) the carried panels re-priced before anything was built on them
print(f"\n(g) inputs: CA panel re-price max |diff| "
      f"{CERTIN['ca_reprice_max_abs_diff_bp']:.10f} bp over "
      f"{CERTIN['ca_reprice_cells']} cells; {CERTIN['rolls']['ca']} CA rolls and "
      f"{CERTIN['rolls']['leg']} leg rolls with {CERTIN['rolls']['common']} in "
      "common")
assert CERTIN["ca_reprice_max_abs_diff_bp"] < 1e-9
assert CERTIN["rolls"]["common"] == 0
print("\nAll seven tie out. The reading of the note's conventions is confirmed "
      "from the note's own printed numbers, and the promoted module from the "
      "reproduction's own recorded path, before any backtest number is read.")

(a) DV01 weights implied by $147mm/-$85.6mm/$20.89mm at a flat 2% curve: 2y 0.7074 vs 0.705 (+0.34%), 10y 0.4651 vs 0.465 (+0.02%)
(b) belly DV01 from the Figure-6 beta $41,200/bp vs from the printed notional $40,347/bp (+2.1%)
(c) carry = 1.3bp x $200,000 + 3.2bp x $41,200 = $391,840 vs the note's $380,000 (+3.1%)
(d) fly -18.2bp -> fitted CA 5.951bp, CA 8.8bp -> rich +2.849bp vs 'about 3bp'
(e) side*(dCA - beta*dfly)*DV01 = -1 * ((6.6-8.8) - 0.206*(-16.5-(-18.2))) * 200,000 = $510,040
    Citi's own table records $552,000 and the alert says 'net +$500k' (-7.6% vs the table)

(f) citi_fv against the reproduction it was lifted from (2022-01-03..2026-08-21, 504 bd):
    b sign flips 1 at the 2023-06-20 refit;  w2 0.05 -> 0.95;  b +19.1 -> -7.6
    OOS residual sd: IMM refit 2.232 bp, Citi's fixed 2017 weights 2.149 bp
    fly starts: best spot (Citi's own) at 2.232 bp; the matched-expiry IMM_13 ranks 9 of 10

(g) inputs: CA panel re-price max |diff| 0.0000000000 bp over 30 cells; 22 CA 

## Figure 20 — the screen, on every date

> *"Figure 20 offers a systematic analysis of the valuation in convexity
> adjustments across the curve. We analyze 1y packs (four consecutive
> contracts) because individual ED/FRA spreads are noisy and hard to trade."*

The note prints one date; the backtest needs it on every date, because the
screen is what chooses which structure to trade.

In [5]:
S = SC.build_screen(P)
_ids = SC.verify_identities(S, P)
print(f"identity  CA - Model - VsModel  max |err| "
      f"{_ids['ca_minus_model_minus_vsmodel']:.3e} bp "
      f"({_ids['n_cells_skipped_vsmodel']} cells skipped: two dates carry no "
      "vol mark)")
print(f"identity  implied^2*w/2e4 == CA  max |err| "
      f"{_ids['implied_reconstructs_ca']:.3e} bp "
      f"({_ids['n_cells_skipped_implied']} skipped: a non-positive CA has no "
      "real implied vol, and the front packs go negative often)")
assert _ids["ca_minus_model_minus_vsmodel"] < 1e-9
assert _ids["implied_reconstructs_ca"] < 1e-6

_last = P.index[-1]
print(f"\nFigure 20 analogue, close of {_last.date()}:\n")
print(SC.screen_table(S, _last).round(2).to_string())
print(f"\nCiti's H0-Z0 row (12-Jan-2017, ED): CA {CFG.fig20_ca_bp}, VsModel "
      f"{CFG.fig20_vs_model_bp}, 3m Roll {CFG.fig20_roll_bp}, Implied "
      f"{CFG.fig20_implied}, Realized {CFG.fig20_realized}, Impl/Rlzd "
      f"{CFG.fig20_implied / CFG.fig20_realized:.1f}")

identity  CA - Model - VsModel  max |err| 0.000e+00 bp (10 cells skipped: two dates carry no vol mark)
identity  implied^2*w/2e4 == CA  max |err| 7.105e-15 bp (997 skipped: a non-positive CA has no real implied vol, and the front packs go negative often)

Figure 20 analogue, close of 2026-08-21:

          ca  chg_1w  ca_z_3m  ca_z_1y  model  vs_model  vs_model_z_3m  vs_model_z_1y  roll_3m  implied  realized  impl_rlzd
WHITES  0.19   -0.25    -0.05     0.10   0.12      0.06          -0.02           0.11     0.10   116.90     62.54       1.87
REDS    0.81    0.74    -0.18     0.67   0.99     -0.18          -0.12           0.64     0.33    86.29     77.70       1.11
GREENS  2.61   -0.14     0.65     0.35   2.73     -0.12           0.73          -0.12     0.55    92.37     72.61       1.27
BLUES   6.50   -0.37     1.66     1.18   5.23      1.27           1.72           0.64     0.75   104.00     70.03       1.49
GOLDS   9.90   -0.07     1.38     1.73   8.31      1.59           1.51       

### The roll column, two ways

The note's `3m Roll` is `CA(p) − CA(p one contract nearer)`. Colour packs are
four contracts — one year — apart, so the adjacent-quarter neighbour does not
exist in the tradeable set. The declared column is the analytic equivalent,
one quarter of the CA's own decay `dCA/dt = −σ²·mean(T1)/1e4`; the
term-structure form `[CA(p) − CA(p one YEAR nearer)]/4` is measured beside it
rather than asserted equal to it.

In [6]:
print(SC.roll_identity(S).round(4).to_string())
_rj = pd.DataFrame(PRE["roll_jump"]).set_index("structure")
print("\nand the same quantity a third way -- the measured IMM-roll JUMP, which "
      "is the theta being paid back:")
print(_rj[["n_rolls", "mean_dCA_on_roll", "t_on_roll", "quarter_theta_bp",
           "jump_over_quarter_theta"]].round(4).to_string())
print("\nGREENS / BLUES / GOLDS agree to 2-11%. WHITES and REDS do not, and "
      "they are the two structures whose daily marks sit at or past the -0.5 "
      "pure-noise bound -- which is why they are out of the primary selection "
      "universe.")

           nearer     n  analytic_mean_bp  term_structure_mean_bp   ratio    corr
structure                                                                        
REDS       WHITES  1407            0.4582                  0.1529  2.9962  0.3830
GREENS       REDS  1407            0.7095                  0.8349  0.8498 -0.0638
BLUES      GREENS  1407            0.9238                  1.1897  0.7765  0.5824
GOLDS       BLUES  1407            1.1005                  1.2005  0.9167  0.7550

and the same quantity a third way -- the measured IMM-roll JUMP, which is the theta being paid back:
           n_rolls  mean_dCA_on_roll  t_on_roll  quarter_theta_bp  jump_over_quarter_theta
structure                                                                                 
WHITES          22            0.7025     0.8025            0.1630                   4.3089
REDS            22           -0.3338    -0.3228            0.4582                  -0.7285
GREENS          22            0.7413     2

## Figure 6 — the fitted fair value, and whether it is stable enough to hedge with

Citi's method: regress the CA on 2y/5y/10y and read the hedge weights off the
fit. Here it is fly-constrained (`w₂ + w₁₀ = 1`, so the fitted object is a
real butterfly), refit at every quarterly SR3 IMM roll on a trailing 504 bd
window ending strictly before the roll, and applied out of sample until the
next roll.

In [7]:
_fvrows = []
for _lab, _d in PRE["fair_value"].items():
    _fvrows.append({"structure": _lab, "n_refits": _d["n_refits"],
                    "w2_median": _d["w2_median"], "w2_range": _d["w2_range"],
                    "b_median": _d["b_median"],
                    "b_sign_flips": _d["b_sign_flips"],
                    "w2_at_a_boundary": _d["n_boundary_w2"],
                    "oos_resid_sd_bp": _d["oos_resid_sd_bp"],
                    "oos_resid_mae_bp": _d["oos_resid_mae_bp"]})
FVT = pd.DataFrame(_fvrows).set_index("structure")
print(FVT.round(3).to_string())
print(f"\nOn the reproduction's narrower window (2022 start) BLUES has ONE sign "
      f"flip and a residual sd of {TIE['residuals']['sd_refit']:.3f} bp. Adding "
      "2021 doubles both. The window was not narrowed to hide that: the panel "
      "is the full CA panel and the instability is a result.")

_f = go.Figure()
for _lab, _d in PRE["fair_value"].items():
    _f.add_trace(go.Scatter(x=pd.to_datetime(_d["in_force_from"]), y=_d["b_path"],
                            name=_lab, mode="lines+markers"))
_f.add_hline(y=0.0, line_dash="dash", line_color="black")
_f.update_layout(title="The fitted scale b, refit at every IMM roll — crossing "
                       "zero is the failure mode",
                 yaxis_title="b (bp of CA per percent of fly)", height=430,
                 legend=dict(orientation="h", y=1.10))
_f.show()
print("A hedge ratio whose SIGN changes inside its own sample cannot be hedged "
      "with, however well it fits on average: the hedge would have to be turned "
      "upside down mid-trade and nothing in the fit says in advance which side "
      "of the flip it is on.")

           n_refits  w2_median  w2_range  b_median  b_sign_flips  w2_at_a_boundary  oos_resid_sd_bp  oos_resid_mae_bp
structure                                                                                                            
GREENS           21       0.31       0.9    -8.946             4                11            2.350             2.332
BLUES            21       0.70       0.9    12.422             4                 9            4.537             4.131
GOLDS            21       0.34       0.9    31.200             2                 6            7.280             6.759

On the reproduction's narrower window (2022 start) BLUES has ONE sign flip and a residual sd of 2.232 bp. Adding 2021 doubles both. The window was not narrowed to hide that: the panel is the full CA panel and the instability is a result.


A hedge ratio whose SIGN changes inside its own sample cannot be hedged with, however well it fits on average: the hedge would have to be turned upside down mid-trade and nothing in the fit says in advance which side of the flip it is on.


## The entry conjunction — and the headline finding

The note's entry is five conditions at once. Each threshold below is the
note's own printed value, frozen in the pre-registration before anything was
scored.

In [8]:
CTX = R.build_contexts(P, S, R.RuleConfig(), structures=SC.SCREEN_STRUCTURES)
CTXP = {k: v for k, v in CTX.items() if k in R.PRIMARY_UNIVERSE}
B = R.condition_binding(CTXP, R.RuleConfig())
print(B[[f"pass_{k}" for k in R.CONDITIONS] + ["pass_all"]].round(4).to_string())
_nest = pd.DataFrame(PROBE["nested"]).set_index("structure")
print("\nnested, one condition at a time (days surviving):")
print(_nest.to_string())
_days = PROBE["days_open_raw"]
print(f"\n**The five-way conjunction is satisfied on {_days} of {len(P)} dates "
      "across the whole primary universe.**")

           pass_wide_to_model  pass_wide_to_fly  pass_positive_roll  pass_implied_rich  pass_positioning_stretched  pass_all
structure                                                                                                                   
GREENS                 0.0248            0.0199              0.9986             0.1923                      0.3534    0.0007
BLUES                  0.0234            0.0234              0.9986             0.2697                      0.3534    0.0007
GOLDS                  0.0284            0.0546              0.9986             0.2229                      0.3534    0.0000

nested, one condition at a time (days surviving):
           after_wide_to_model  after_wide_to_fly  after_positive_roll  after_implied_rich  after_positioning_stretched
structure                                                                                                              
GREENS                      35                  6                    6              

### Why: two of the note's own conditions fight the first one

The pairwise **lift** is the observed joint pass rate divided by the rate
under independence. Above one, two conditions agree; below one, they fight.

In [9]:
LIFT = pd.DataFrame(PROBE["pair_lift_blues"])
LIFT = LIFT.loc[list(R.CONDITIONS), list(R.CONDITIONS)]
print(LIFT.round(3).to_string())
print("\n* `wide_to_model` x `implied_rich` = "
      f"{LIFT.loc['wide_to_model', 'implied_rich']:.2f}: a CA that is unusually "
      "wide to the Ho-Lee model tends to occur when realised vol has been HIGH, "
      "so implied/realised is low at exactly the moment the model says the "
      "adjustment is rich.")
print("* `wide_to_model` x `positioning_stretched` = "
      f"{LIFT.loc['wide_to_model', 'positioning_stretched']:.2f}: the note's "
      "causal chain -- dealers long futures -> structurally short CA -> wider "
      "CA -- runs the OTHER WAY on SOFR. Measured separately: "
      f"corr(BLUES CA-vs-model, dealer 1Y z) = "
      f"{CERTIN['cftc']['corr_vsmodel_vs_dealerz']:+.3f}.")
print("* `wide_to_model` x `wide_to_fly` = "
      f"{LIFT.loc['wide_to_model', 'wide_to_fly']:.2f}: better than "
      "independence, but the two 'wideness' measures the note treats as saying "
      "the same thing pass 2.3% of days each and only 0.14% jointly.")

                       wide_to_model  wide_to_fly  positive_roll  implied_rich  positioning_stretched
wide_to_model                 42.697        2.588          1.001         0.337                  0.429
wide_to_fly                    2.588       42.697          1.001         0.899                  1.029
positive_roll                  1.001        1.001          1.001         0.999                  1.001
implied_rich                   0.337        0.899          0.999         3.708                  1.013
positioning_stretched          0.429        1.029          1.001         1.013                  2.829

* `wide_to_model` x `implied_rich` = 0.34: a CA that is unusually wide to the Ho-Lee model tends to occur when realised vol has been HIGH, so implied/realised is low at exactly the moment the model says the adjustment is rich.
* `wide_to_model` x `positioning_stretched` = 0.43: the note's causal chain -- dealers long futures -> structurally short CA -> wider CA -- runs the OTHER WAY o

### The declared threshold ladder

A rule with two entry days cannot be graded, so the pre-registration declares
**two rungs** — the note's own 2.0σ and a widened 1.0σ — and prints the rungs
it does not walk so a reader can see the ladder rather than wonder about it.

In [10]:
LAD = pd.DataFrame(PROBE["ladder"])
print(LAD.pivot(index="z_min", columns="impl_rlzd_min",
                values="days_any_structure").to_string())
print("\nrows are the z threshold on BOTH wideness conditions, columns the "
      "impl/rlzd floor; the cell is the number of dates on which SOME primary "
      f"structure passes all five. **Scored: z = {CFG.z_levels[0]} and "
      f"z = {CFG.z_levels[1]} at impl/rlzd {CFG.impl_rlzd_min}. NOT scored: "
      "1.5, 0.5 and 0.0.**")

impl_rlzd_min  0.0  1.0  1.3
z_min                       
0.0            161  108   51
0.5             84   59   30
1.0             20   18   11
1.5              4    4    2
2.0              2    2    2

rows are the z threshold on BOTH wideness conditions, columns the impl/rlzd floor; the cell is the number of dates on which SOME primary structure passes all five. **Scored: z = 2.0 and z = 1.0 at impl/rlzd 1.3. NOT scored: 1.5, 0.5 and 0.0.**


In [11]:
_f = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.62, 0.38],
                   vertical_spacing=0.07,
                   subplot_titles=("BLUES convexity adjustment, its Ho-Lee "
                                   "model level and its fitted fly",
                                   "the two wideness z-scores, and the 2σ line"))
_f.add_trace(go.Scatter(x=P.index, y=CTX["BLUES"].ca_pnl * 0
                        + P["blues_ca_bp"], name="Blues CA"), row=1, col=1)
_f.add_trace(go.Scatter(x=P.index, y=S["model"]["BLUES"], name="Ho-Lee model"),
             row=1, col=1)
_f.add_trace(go.Scatter(x=CTX["BLUES"].fitted.dropna().index,
                        y=CTX["BLUES"].fitted.dropna(),
                        name="fitted 2s5s10s fly (OOS)"), row=1, col=1)
_f.add_trace(go.Scatter(x=P.index, y=CTX["BLUES"].z_model, name="z vs model",
                        line=dict(color="#4dabf7")), row=2, col=1)
_f.add_trace(go.Scatter(x=P.index, y=CTX["BLUES"].z_fly, name="z vs fly",
                        line=dict(color="#ff9f43")), row=2, col=1)
_f.add_hline(y=2.0, line_dash="dot", row=2, col=1)
_f.add_hline(y=1.0, line_dash="dash", row=2, col=1)
_f.update_yaxes(title_text="bp", row=1, col=1)
_f.update_yaxes(title_text="sigma", row=2, col=1)
_f.update_layout(height=640, legend=dict(orientation="h", y=1.06))
_f.show()

## The declared grid

23 cells, one headline. The bar is `E[max SR | null]` at 23 trials on the
**tradeable** span, not the panel span: a book that cannot open until its fair
value and its z-scores exist has not been running for the whole panel.

In [12]:
_cols = ["cell_id", "tier", "headline", "hedge", "selection", "n_conditions",
         "n_episodes", "mean_hold_bd", "n_eff", "hit_rate", "exit_target",
         "exit_stop", "exit_max_hold", "exit_end_of_sample", "net_0.0",
         "sharpe_0.0", "net_1.0", "sharpe_1.0", "net_2.0", "sharpe_2.0",
         "breakeven_bp", "carry_share"]
print(STATS[_cols].round(4).to_string(index=False))
_alive = STATS[STATS["sharpe_1.0"] > BAR]
print(f"\nE[max SR | null] at {CFG.n_declared} trials, span {SPAN:.3f} y: "
      f"**{BAR:.4f}** annualised "
      f"({GMETA['null_bars']['23']['emax_perhold']:.4f} per-hold)")
print(f"cells clearing it at 1x costs: **{len(_alive)} of {len(STATS)}**")
print(f"best net Sharpe on the panel: {STATS['sharpe_1.0'].max():+.4f} "
      f"({STATS.loc[STATS['sharpe_1.0'].idxmax(), 'cell_id']})")
print(f"best GROSS Sharpe: {STATS['sharpe_0.0'].max():+.4f} "
      f"({STATS.loc[STATS['sharpe_0.0'].idxmax(), 'cell_id']})")
assert len(_alive) == GMETA["n_alive_1x"]
assert STATS["sharpe_0.0"].max() < BAR, (
    "a cell clears the bar even GROSS -- the verdict prose below is wrong")

                             cell_id       tier  headline         hedge        selection  n_conditions  n_episodes  mean_hold_bd  n_eff  hit_rate  exit_target  exit_stop  exit_max_hold  exit_end_of_sample      net_0.0  sharpe_0.0       net_1.0  sharpe_1.0       net_2.0  sharpe_2.0  breakeven_bp  carry_share
     P|z2.0|screen_best|fitted_refit    primary      True  fitted_refit      screen_best             5           2        5.5000    2.0    0.5000            0          1              0                   1 -874329.1711     -0.3859 -1.189942e+06     -0.5177 -1.505555e+06     -0.6152       -1.0519      -0.0280
    P|z2.0|screen_best|fitted_frozen    primary     False fitted_frozen      screen_best             5           2        5.5000    2.0    0.5000            0          1              0                   1 -874329.1711     -0.3859 -1.189942e+06     -0.5177 -1.505555e+06     -0.6152       -1.0519      -0.0280
        P|z2.0|screen_best|citi_2017    primary     False     citi_2017  

### The headline — the note's own trade at the note's own thresholds

In [13]:
_h = STATS[STATS["headline"]].iloc[0]
_he = EPS[EPS.cell_id == _h["cell_id"]]
print(f"{_h['cell_id']}\n")
print(_he[["structure", "entry_decision", "entry_fill", "exit_fill", "hold_bd",
           "beta_entry", "w2_entry", "exit_reason", "z_model", "z_fly",
           "rich_bp", "impl_rlzd", "z_pos", "gross_usd"]].round(3).to_string(index=False))
print(f"\ntwo episodes in {SPAN:.1f} years. Net at 1x costs "
      f"${_h['net_1.0']:,.0f}, annualised Sharpe {_h['sharpe_1.0']:+.4f} "
      f"against a bar of {BAR:.4f}.")
print("\nThe first is the March-2023 regional-bank week: the Blues CA was "
      f"{_he.iloc[0]['rich_bp']:.1f}bp rich to its own fitted fly at "
      f"{_he.iloc[0]['z_model']:.2f} sigma to the model -- exactly the setup the "
      "note describes -- and it widened further. The stop did its job and the "
      "trade lost "
      f"${abs(_he.iloc[0]['gross_usd']):,.0f} in {int(_he.iloc[0]['hold_bd'])} "
      "business days. The second opened two weeks before the end of the sample "
      "and is still open.")

P|z2.0|screen_best|fitted_refit

structure entry_decision entry_fill  exit_fill  hold_bd  beta_entry  w2_entry   exit_reason  z_model  z_fly  rich_bp  impl_rlzd  z_pos   gross_usd
    BLUES     2023-03-15 2023-03-16 2023-03-20        2      -0.033      0.95          stop    3.909  2.666   10.367      1.400  1.499 -880772.546
   GREENS     2026-08-07 2026-08-10 2026-08-21        9       0.045      0.05 end_of_sample    2.374  2.085    0.935      1.366  1.838    6443.375

two episodes in 4.7 years. Net at 1x costs $-1,189,942, annualised Sharpe -0.5177 against a bar of 0.9066.

The first is the March-2023 regional-bank week: the Blues CA was 10.4bp rich to its own fitted fly at 3.91 sigma to the model -- exactly the setup the note describes -- and it widened further. The stop did its job and the trade lost $880,773 in 2 business days. The second opened two weeks before the end of the sample and is still open.


## The second clock — per-hold, not annualised

The pre-registration requires `E[max SR | null]` on **both** clocks (§6, §11)
and the grid above grades only the annualised one. For a book that is flat on
99% of its dates the two are not interchangeable:

* the **annualised daily** Sharpe divides by the sd of a series that is mostly
  zeros, so it measures the equity curve an investor would actually hold, idle
  capital included. Four trades over 1,409 dates score low almost by
  construction;
* the **per-hold** Sharpe is `mean / sd` over the EPISODES and measures the
  quality of the trades that were taken. Its null sd is `1/sqrt(n_eff)`, which
  is exactly what `null_bars(..., n_eff=...)["emax_perhold"]` returns.

Grading either against the other's bar is the "which column is the claim true
in" error. Amendment A1 of the pre-registration records that this grading was
computed after the grid ran; it adds no cells and no trials.

In [14]:
PH = pd.read_parquet(DATA / "p4_perhold.parquet")
_pc = ["cell_id", "tier", "headline", "n", "n_eff", "trades_per_year",
       "perhold_sharpe_gross", "perhold_sharpe_net", "bar_perhold",
       "clears_perhold_gross", "clears_perhold_net", "p_signflip_gross",
       "p_signflip_net"]
print(PH[_pc].round(4).to_string(index=False))
_cg = PH[PH["clears_perhold_gross"].astype("boolean").fillna(False).astype(bool)]
_cn = PH[PH["clears_perhold_net"].astype("boolean").fillna(False).astype(bool)]
print(f"\ncells clearing their own per-hold bar GROSS: {len(_cg)} of {len(PH)}")
print(f"cells clearing it NET of 1x costs:          {len(_cn)} of {len(PH)}")
print(f"best per-hold Sharpe gross {PH['perhold_sharpe_gross'].max():+.4f}, "
      f"net {PH['perhold_sharpe_net'].max():+.4f}")
print(f"best shared-sign-flip p on NET per-episode P&L: "
      f"{PH['p_signflip_net'].min():.4f}")
assert len(_cn) == 0, "a cell clears the per-hold bar NET -- the verdict changes"

                             cell_id       tier  headline  n  n_eff  trades_per_year  perhold_sharpe_gross  perhold_sharpe_net  bar_perhold clears_perhold_gross clears_perhold_net  p_signflip_gross  p_signflip_net
     P|z2.0|screen_best|fitted_refit    primary      True  2    2.0           0.4272               -0.6968             -0.9509       1.3870                False              False               NaN             NaN
    P|z2.0|screen_best|fitted_frozen    primary     False  2    2.0           0.4272               -0.6968             -0.9509       1.3870                False              False               NaN             NaN
        P|z2.0|screen_best|citi_2017    primary     False  1    1.0           0.2136                   NaN                 NaN       1.9615                False              False               NaN             NaN
         P|z2.0|screen_best|unhedged    primary     False  2    2.0           0.4272               -0.6668             -0.9066       1.3870     

### Read the two clocks together

Apart they say different things and both are true. The framework selects
trades that are better than chance *gross* and cannot pay for them; and the
book it produces is far too sparse to run whatever the trades are worth.

In [15]:
_join = STATS[["cell_id", "sharpe_0.0", "sharpe_1.0"]].merge(
    PH[["cell_id", "n", "perhold_sharpe_gross", "perhold_sharpe_net",
        "bar_perhold"]], on="cell_id", how="left")
_join["bar_annualised"] = BAR
_join["clears_annualised_gross"] = _join["sharpe_0.0"] > BAR
_join["clears_perhold_gross"] = (_join["perhold_sharpe_gross"]
                                 > _join["bar_perhold"])
print(_join.round(4).to_string(index=False))
print(f"\nannualised clock: {int(_join['clears_annualised_gross'].sum())} of "
      f"{len(_join)} clear GROSS, "
      f"{int((STATS['sharpe_1.0'] > BAR).sum())} clear NET")
print(f"per-hold clock:   {int(_join['clears_perhold_gross'].fillna(False).sum())} "
      f"of {len(_join)} clear GROSS, {len(_cn)} clear NET")
_a = float(PH.loc[PH.cell_id == "D|z1.0|drop_positive_roll", "net_usd"].iloc[0])
_b = float(PH.loc[PH.cell_id == "P|z1.0|screen_best|fitted_refit",
                  "net_usd"].iloc[0])
assert abs(_a - _b) < 1e-6, "the drop_positive_roll claim below is wrong"
print(f"\nThree of the {len(_cg)} gross-clearing cells are drop-one "
      "DIAGNOSTICS, and D|z1.0|drop_positive_roll is IDENTICAL to "
      f"P|z1.0|screen_best|fitted_refit (both ${_a:,.0f} net) because the "
      "positive-roll condition passes on 99.86% of dates and the days it "
      "refuses never coincide with the other four passing. What is left is the "
      "screen_best z=1.0 family, on four trades, whose own per-hold null sd is "
      "0.5 -- which is why its bar is 0.98.")

                             cell_id  sharpe_0.0  sharpe_1.0  n  perhold_sharpe_gross  perhold_sharpe_net  bar_perhold  bar_annualised  clears_annualised_gross  clears_perhold_gross
     P|z2.0|screen_best|fitted_refit     -0.3859     -0.5177  2               -0.6968             -0.9509       1.3870          0.9066                    False                 False
    P|z2.0|screen_best|fitted_frozen     -0.3859     -0.5177  2               -0.6968             -0.9509       1.3870          0.9066                    False                 False
        P|z2.0|screen_best|citi_2017      0.1088     -0.3821  1                   NaN                 NaN       1.9615          0.9066                    False                 False
         P|z2.0|screen_best|unhedged     -0.3738     -0.5027  2               -0.6668             -0.9066       1.3870          0.9066                    False                 False
           P|z2.0|blues|fitted_refit         NaN         NaN  1                   NaN     

## Engine certification

The panel decides WHEN; the engine says what it was worth. Every reported book
is replayed through `QueryDrivenBacktest` on dated instruments — four SR3
contracts, a matched-maturity quarterly/quarterly swap, and a spot 2s5s10s fly
at the fitted weights re-struck at each quarterly refit inside the hold — with
`assert_ran` afterwards, because `run()` swallows exceptions and a failed
backtest looks like a flat equity curve.

In [16]:
_ec = ["cell_id", "n_episodes", "n_closed_positions", "n_marks",
       "panel_gross_usd", "engine_gross_usd", "panel_net_usd", "engine_net_usd",
       "panel_sharpe_net", "engine_sharpe_net", "daily_corr_engine_panel",
       "carry_usd", "engine_residual_usd"]
E = ENG[_ec].copy()
E["engine_over_panel_gross"] = E["engine_gross_usd"] / E["panel_gross_usd"]
print(E.round(4).to_string(index=False))
print(f"\nbest ENGINE net Sharpe of any certified book: "
      f"{ENG['engine_sharpe_net'].max():+.4f} "
      f"({ENG.loc[ENG['engine_sharpe_net'].idxmax(), 'cell_id']}), against a "
      f"bar of {BAR:.4f}.")
print("\nThe gap between the two columns is the point of running both. It is "
      "not noise and it does not have one sign: on "
      f"{(E['engine_over_panel_gross'] < 0).sum()} of {len(E)} books the engine "
      "and the panel disagree on the SIGN of the gross P&L, and the largest "
      "single disagreement is "
      f"{E.loc[E['engine_over_panel_gross'].abs().idxmax(), 'cell_id']} at a "
      f"ratio of {E['engine_over_panel_gross'].abs().max():.2f}. A par-rate "
      "panel prices par-rate changes; the engine prices struck instruments that "
      "age, and the omitted term is carry.")

                             cell_id  n_episodes  n_closed_positions  n_marks  panel_gross_usd  engine_gross_usd  panel_net_usd  engine_net_usd  panel_sharpe_net  engine_sharpe_net  daily_corr_engine_panel   carry_usd  engine_residual_usd  engine_over_panel_gross
     P|z2.0|screen_best|fitted_refit           2                  12      865     -874329.1711      -537512.6449  -1.189942e+06   -8.531255e+05           -0.5177            -0.1701                   0.8798  24464.7178         -561977.3627                   0.6148
           P|z2.0|blues|fitted_refit           1                   6       13     -880772.5465      -931143.6242  -1.037404e+06   -1.087775e+06               NaN                NaN                   0.9261   8943.4594         -940087.0836                   1.0572
     P|z1.0|screen_best|fitted_refit           4                  25     1174     1322847.4465      1300078.7931   5.895245e+05    5.759019e+05            0.1226             0.0681                   0.8111 11

In [17]:
_f = go.Figure()
for _c in EQ.columns:
    _f.add_trace(go.Scatter(x=pd.to_datetime(EQ.index), y=EQ[_c], name=_c,
                            mode="lines"))
_f.add_hline(y=0.0, line_dash="dot")
_f.update_layout(title="Engine equity, net of the declared costs — every "
                       "certified book",
                 yaxis_title="USD", height=520,
                 legend=dict(orientation="h", y=-0.25))
_f.show()

## The control battery

None of these is a scored cell; each is a control on a cell already scored,
and every one is declared in the pre-registration.

### 1. Same-day fills — the mark-noise harvest

In [18]:
_sd = pd.DataFrame(CTRL["same_day"])
_w = _sd.pivot(index="cell_id", columns="exec_lag_bd", values="gross_usd")
_w["harvest_usd"] = _w[0] - _w[1]
print(_w.round(0).to_string())
print(f"\nEvery one of the {len(_w)} cells is worth "
      f"${_w['harvest_usd'].min():,.0f} to ${_w['harvest_usd'].max():,.0f} MORE "
      "when it is filled at the mark its own signal was computed from, and the "
      f"harvest is positive on {int((_w['harvest_usd'] > 0).sum())} of "
      f"{len(_w)}. That gap is not a strategy: the CA mark is a composite of a "
      "futures bar and a separately-timed swap curve, and a rule that shorts a "
      "rich mark AT that mark banks a reversion nobody can trade. It is the "
      "single best argument for the `exec_lag_bd = 1` convention, which every "
      "number in this notebook uses.")
assert (_w["harvest_usd"] > 0).all()

exec_lag_bd                                 0          1  harvest_usd
cell_id                                                              
P|z1.0|blues|fitted_refit           3052895.0  1142551.0    1910343.0
P|z1.0|screen_best|citi_2017        1254457.0  -653422.0    1907880.0
P|z1.0|screen_best|fitted_refit     3236151.0  1322847.0    1913303.0
P|z1.0|screen_best|unhedged         3102301.0  1221105.0    1881196.0
P|z2.0|screen_best|fitted_refit     1567511.0  -874329.0    2441840.0
S|z1.0|screen_best_all5|citi_2017  24773669.0  7012312.0   17761357.0

Every one of the 6 cells is worth $1,881,196 to $17,761,357 MORE when it is filled at the mark its own signal was computed from, and the harvest is positive on 6 of 6. That gap is not a strategy: the CA mark is a composite of a futures bar and a separately-timed swap curve, and a rule that shorts a rich mark AT that mark banks a reversion nobody can trade. It is the single best argument for the `exec_lag_bd = 1` convention, which every

### 2. The placebo ladder — a timing signal must die under lag

In [19]:
_pl = pd.DataFrame(CTRL["placebo"])
print(_pl.pivot(index="cell_id", columns="signal_lag_bd",
                values="sharpe_gross").round(4).to_string())
print("\ngross $ at each lag:")
print(_pl.pivot(index="cell_id", columns="signal_lag_bd",
                values="gross_usd").round(0).to_string())
_pw = _pl.pivot(index="cell_id", columns="signal_lag_bd", values="sharpe_gross")
_live = _pw[_pw[0] > 0]
print(f"\nOf the {len(_pw)} cells, {len(_live)} make money unlagged. Every "
      f"one of those {len(_live)} is NEGATIVE by 40 bd of lag "
      f"(max {_live[40].max():+.4f}), and the median decay from lag 0 to lag "
      f"60 is {float((_live[0] - _live[60]).median()):+.4f} of Sharpe. So the "
      "small edge that is there is genuinely about WHEN rather than a slow "
      "level effect -- it is just an order of magnitude below the bar.")
assert (_live[40] < 0).all()

signal_lag_bd                          0       10      20      40      60
cell_id                                                                  
P|z1.0|blues|fitted_refit          0.2516  0.2273 -0.2148 -0.1818 -0.3792
P|z1.0|screen_best|citi_2017      -0.1856  0.0321 -0.1496 -0.2516 -0.1902
P|z1.0|screen_best|fitted_refit    0.2989  0.2389 -0.1777 -0.1645 -0.3938
P|z1.0|screen_best|unhedged        0.2079  0.0746 -0.0877 -0.1635 -0.2340
P|z2.0|screen_best|fitted_refit   -0.3859 -0.0530     NaN  0.1455  0.2695
S|z1.0|screen_best_all5|citi_2017  0.4155 -0.2636 -0.0261 -0.0118  0.2764

gross $ at each lag:
signal_lag_bd                             0          10        20        40         60
cell_id                                                                               
P|z1.0|blues|fitted_refit          1142551.0   726750.0 -883314.0 -713425.0 -1544384.0
P|z1.0|screen_best|citi_2017       -653422.0    94594.0 -648932.0 -821248.0  -704270.0
P|z1.0|screen_best|fitted_refit    132

### 3. The always-short control — how much of the book is a static position?

In [20]:
_as = pd.DataFrame(CTRL["always_short"])
print(_as.round(0).to_string(index=False))
_sd2 = pd.DataFrame(CTRL["always_short"])
_raw = pd.read_parquet(DATA / "p4_controls_static.parquet")
print(f"\nThe static short is profitable on its own on "
      f"{int((_raw['per_trade_usd'] > 0).sum())} of {len(_raw)} "
      f"(cell, structure) pairs, at ${_raw['per_trade_usd'].min():,.0f} to "
      f"${_raw['per_trade_usd'].max():,.0f} per trade: a short-convexity book "
      "collects the CA's theta whether or not a signal fired. The signal's "
      "own contribution is the last column above, and it is positive on "
      f"{int((_sd2['signal_edge_usd'] > 0).sum())} of {len(_sd2)} cells -- on "
      "four trades each.")

                          cell_id  per_signal_trade_usd  per_static_trade_usd  signal_edge_usd
  P|z2.0|screen_best|fitted_refit             -437165.0                3969.0        -441134.0
  P|z1.0|screen_best|fitted_refit              330712.0               28301.0         302410.0
     P|z1.0|screen_best|citi_2017             -130684.0               37761.0        -168445.0
      P|z1.0|screen_best|unhedged              305276.0               99484.0         205792.0
        P|z1.0|blues|fitted_refit              285638.0               14674.0         270964.0
S|z1.0|screen_best_all5|citi_2017              318741.0               22065.0         296677.0

The static short is profitable on its own on 15 of 16 (cell, structure) pairs, at $-3,506 to $149,112 per trade: a short-convexity book collects the CA's theta whether or not a signal fired. The signal's own contribution is the last column above, and it is positive on 4 of 6 cells -- on four trades each.


### 4. β = 0 — does the fly leg contribute?

In [21]:
print(pd.DataFrame(CTRL["beta_zero"]).round(4).to_string(index=False))
_b0 = pd.DataFrame(CTRL["beta_zero"])
_worst = _b0.loc[_b0["hedge_adds_sharpe"].idxmin()]
print(f"\nThe fly adds Sharpe on {int((_b0['hedge_adds_sharpe'] > 0).sum())} "
      f"of {len(_b0)} and removes it on "
      f"{int((_b0['hedge_adds_sharpe'] < 0).sum())}. The worst is "
      f"{_worst['cell_id']} at {_worst['hedge_adds_sharpe']:+.4f} of Sharpe "
      "against the same book with the hedge removed -- and that is Citi's "
      "own published fixed weights, held fixed.")

                          cell_id  n_hedged  n_beta0  gross_hedged  gross_beta0  sharpe_hedged  sharpe_beta0  hedge_adds_sharpe
  P|z2.0|screen_best|fitted_refit         2        2  -874329.1711 -834043.2707        -0.3859       -0.3738            -0.0121
  P|z1.0|screen_best|fitted_refit         4        4  1322847.4465 1221105.0823         0.2989        0.2079             0.0911
     P|z1.0|screen_best|citi_2017         5        4  -653422.4622 1221105.0823        -0.1856        0.2079            -0.3935
        P|z1.0|blues|fitted_refit         4        4  1142551.3502  974974.5443         0.2516        0.1764             0.0752
S|z1.0|screen_best_all5|citi_2017        22       20  7012312.1970 6879188.3767         0.4155        0.4227            -0.0072

The fly adds Sharpe on 2 of 5 and removes it on 3. The worst is P|z1.0|screen_best|citi_2017 at -0.3935 of Sharpe against the same book with the hedge removed -- and that is Citi's own published fixed weights, held fixed.


### 5. The splice control

In [22]:
print(pd.DataFrame(CTRL["splice"]["roll_artifact"]).round(6).to_string(index=False))
_sp2 = pd.DataFrame(CTRL["splice"]["books"])
print("\nthe same books with the P&L splice off (i.e. booking the roll jump):")
print(_sp2.pivot(index="cell_id", columns="splice_pnl",
                 values="gross_usd").round(0).to_string())
print("\nThe splice is switched on: the raw series carries +0.95 bp per roll on "
      "BLUES and the spliced one carries exactly zero. Booking that jump would "
      "cost a short-CA book real money 22 times -- and it would be an artifact, "
      "because a DATED position has no label to switch.")

structure  raw_mean_dCA_on_roll  spliced_mean_dCA_on_roll  n_rolls
   GREENS              0.741295                      -0.0       22
    BLUES              0.945976                       0.0       22
    GOLDS              1.222770                       0.0       22

the same books with the P&L splice off (i.e. booking the roll jump):
splice_pnl                             False      True 
cell_id                                                
P|z1.0|blues|fitted_refit           975473.0  1142551.0
P|z1.0|screen_best|citi_2017       -653422.0  -653422.0
P|z1.0|screen_best|fitted_refit    1155769.0  1322847.0
P|z1.0|screen_best|unhedged         671585.0  1221105.0
P|z2.0|screen_best|fitted_refit    -874329.0  -874329.0
S|z1.0|screen_best_all5|citi_2017  2822318.0  7012312.0

The splice is switched on: the raw series carries +0.95 bp per roll on BLUES and the spliced one carries exactly zero. Booking that jump would cost a short-CA book real money 22 times -- and it would be an artifac

### 6. The sign-flip null

In [23]:
print(pd.DataFrame(CTRL["signflip"]).round(4).to_string(index=False))
print("\nA row permutation leaves a Sharpe unchanged, so the null is 20,000 "
      "SHARED SIGN FLIPS on the per-episode P&L. At 4-23 episodes the smallest "
      "attainable one-sided p is 2^-n, so these are bounds on the evidence "
      "rather than measurements of it. One cell -- the secondary "
      "`screen_best_all5 x citi_2017` -- reads p = 0.024 one-sided. It is "
      "16-of-22 WHITES episodes, on the marks this package has measured at or "
      "past the pure-noise bound, and its Sharpe is still less than a quarter "
      "of the bar. It is recorded rather than promoted.")

                          cell_id  n_episodes   t_obs  p_signflip  p_two_sided
  P|z2.0|screen_best|fitted_refit           2     NaN         NaN          NaN
  P|z1.0|screen_best|fitted_refit           4  2.6692      0.0607       0.1230
     P|z1.0|screen_best|citi_2017           5 -0.5350      0.7128       0.6275
      P|z1.0|screen_best|unhedged           4  2.1423      0.0629       0.1272
        P|z1.0|blues|fitted_refit           4  1.8542      0.1276       0.2549
S|z1.0|screen_best_all5|citi_2017          22  2.0773      0.0244       0.0497

A row permutation leaves a Sharpe unchanged, so the null is 20,000 SHARED SIGN FLIPS on the per-episode P&L. At 4-23 episodes the smallest attainable one-sided p is 2^-n, so these are bounds on the evidence rather than measurements of it. One cell -- the secondary `screen_best_all5 x citi_2017` -- reads p = 0.024 one-sided. It is 16-of-22 WHITES episodes, on the marks this package has measured at or past the pure-noise bound, and its Sharpe i

### 7. The convexity signature, as a point prediction

In [24]:
print(pd.DataFrame(CTRL["convexity_signature"]).round(4).to_string(index=False))
print("\nThe CA is exactly quadratic in sigma, so a convexity claim has a "
      "NUMBER attached: the fitted quadratic coefficient must equal "
      "`CA_DV01 * w / 2e4` USD per (bp/yr)^2. At four to twenty-two episodes "
      "this is a three-parameter regression on a handful of points and the "
      "fitted coefficients run 9-51x the prediction with the wrong sign half "
      "the time. The test cannot confirm or deny the claim at this sample size, "
      "and the honest reading is that the book never got large enough to have "
      "a convexity signature to test.")

                          cell_id  n  quad_coef_fitted  quad_coef_predicted  linear_coef    ratio
  P|z1.0|screen_best|fitted_refit  4         1262.3310             128.8715   14272.3151   9.7953
     P|z1.0|screen_best|citi_2017  5        -3243.0395             147.9257   59540.7743 -21.9234
      P|z1.0|screen_best|unhedged  4        -1621.3863             128.8715    6144.2210 -12.5814
        P|z1.0|blues|fitted_refit  4         1094.5413             123.7349    8207.7138   8.8459
S|z1.0|screen_best_all5|citi_2017 22         -962.7558              18.9158   24293.0071 -50.8969

The CA is exactly quadratic in sigma, so a convexity claim has a NUMBER attached: the fitted quadratic coefficient must equal `CA_DV01 * w / 2e4` USD per (bp/yr)^2. At four to twenty-two episodes this is a three-parameter regression on a handful of points and the fitted coefficients run 9-51x the prediction with the wrong sign half the time. The test cannot confirm or deny the claim at this sample size, and 

### 8. Sub-period split, and the sensitivities

In [25]:
_sub = pd.DataFrame(CTRL["subperiod"])
print(_sub.pivot(index="cell_id", columns="half",
                 values=["n_episodes", "gross_usd"]).round(0).to_string())
print("\nsensitivities on P|z1.0|screen_best|fitted_refit (reported, not scored):")
print(pd.DataFrame(CTRL["sensitivity"]).round(4).to_string(index=False))

                                  n_episodes         gross_usd           
half                                   first second      first     second
cell_id                                                                  
P|z1.0|blues|fitted_refit                2.0    2.0   739551.0   403000.0
P|z1.0|screen_best|citi_2017             2.0    3.0 -1223526.0   570103.0
P|z1.0|screen_best|fitted_refit          2.0    2.0   739551.0   583296.0
P|z1.0|screen_best|unhedged              2.0    2.0  1029357.0   191748.0
P|z2.0|screen_best|fitted_refit          1.0    1.0  -880773.0     6443.0
S|z1.0|screen_best_all5|citi_2017       15.0    7.0  5525664.0  1486648.0

sensitivities on P|z1.0|screen_best|fitted_refit (reported, not scored):
         knob value  n_episodes    gross_usd      net_usd  sharpe_gross  sharpe_net
impl_rlzd_min   1.0           7 2320487.5257  918144.6656        0.3659      0.1387
impl_rlzd_min   1.3           4 1322847.4465  589524.5333        0.2989      0.1226
impl_rlz

## The book, as a book

`BT.trade_dashboard` on the episodes of the two cells worth looking at, and
the equity curves side by side. `span_years` is passed explicitly — without
it the annualised Sharpe is wrong.

In [26]:
_HEAD = "P|z1.0|screen_best|fitted_refit"
_bk = EPS[EPS.cell_id == _HEAD].copy()
if len(_bk):
    trade_dashboard(_bk, title=f"{_HEAD} — gross, per episode",
                    span_years=SPAN, time_col="exit_fill", pnl_col="gross_usd",
                    signal_col="z_model", colour_col="exit_reason",
                    label_col="structure", side_col="side", unit="USD").show()

In [27]:
_books = {}
for _c in ("P|z2.0|screen_best|fitted_refit", "P|z1.0|screen_best|fitted_refit",
           "P|z1.0|screen_best|unhedged", "S|z1.0|screen_best_all5|citi_2017"):
    _b = EPS[EPS.cell_id == _c]
    if len(_b):
        _books[_c] = _b
if _books:
    compare_curves(_books, title="declared cells — cumulative gross P&L per "
                                 "episode", time_col="exit_fill",
                   pnl_col="gross_usd", unit="USD").show()

## Verdict

In [28]:
_best_panel = STATS["sharpe_1.0"].max()
_best_eng = ENG["engine_sharpe_net"].max()
print("VERDICT")
print("=" * 78)
print(f"* The rule AS PUBLISHED does not trade. At the note's own thresholds "
      f"the five-way conjunction is satisfied on {_days} of {len(P)} dates "
      f"across GREENS/BLUES/GOLDS, and the headline cell opens twice in "
      f"{SPAN:.1f} years for a net of ${_h['net_1.0']:,.0f}.")
print(f"* The reason is measurable and it is in the note's own conditions. "
      f"'Wide to the model' fights 'implied rich' (lift "
      f"{LIFT.loc['wide_to_model', 'implied_rich']:.2f}) and 'positioning "
      f"stretched' (lift "
      f"{LIFT.loc['wide_to_model', 'positioning_stretched']:.2f}) on SOFR, and "
      "the note's own positioning mechanism runs the other way here "
      f"(corr {CERTIN['cftc']['corr_vsmodel_vs_dealerz']:+.3f}).")
print(f"* Widened to 1 sigma the framework trades 4-23 times. On the "
      f"ANNUALISED clock it still fails: best net Sharpe on the panel "
      f"{_best_panel:+.4f}, best on the ENGINE {_best_eng:+.4f}, against "
      f"E[max SR | null] = {BAR:.4f} at {CFG.n_declared} declared trials over "
      f"{SPAN:.2f} tradeable years, and {len(_alive)} of {len(STATS)} cells "
      "clear it at any cost level.")
print(f"* On the PER-HOLD clock it is not nothing GROSS and is nothing NET. "
      f"{len(_cg)} of {len(PH)} cells clear their own per-hold bar gross -- "
      f"best {PH['perhold_sharpe_gross'].max():+.4f} against a bar of "
      f"{float(PH.loc[PH['perhold_sharpe_gross'].idxmax(), 'bar_perhold']):.4f} "
      f"on four trades -- and {len(_cn)} of {len(PH)} clear it net of 1x costs. "
      f"The best shared-sign-flip p on NET per-episode P&L in the whole block "
      f"is {PH['p_signflip_net'].min():.3f}. So the framework selects trades "
      "that are better than chance and cannot pay for them.")
print(f"* Costs ARE the marginal issue on the per-hold clock -- they take "
      f"{len(_cg)} cells to {len(_cn)} on their own. Break-even runs "
      f"{STATS['breakeven_bp'].abs().median():.2f} bp of gross DV01 traded at "
      "the median, against a declared 0.75 bp on the CA package alone before "
      "the fly's three legs are charged at all.")
print("* Citi's own fair value is not stable enough to hedge with on this "
      f"window: {int(FVT['b_sign_flips'].max())} sign reversals of b in "
      f"{int(FVT['n_refits'].max())} refits, and w2 pinned at a grid boundary "
      f"on {int(FVT['w2_at_a_boundary'].max())} of them.")
_pos = STATS.loc[STATS["net_0.0"] > 0, "carry_share"]
print(f"* What the trade IS, when it works, is short-convexity carry. The "
      f"always-short control is profitable on "
      f"{int((_raw['per_trade_usd'] > 0).sum())} of {len(_raw)} "
      f"(cell, structure) pairs with the signal switched OFF, up to "
      f"${_raw['per_trade_usd'].max():,.0f} per trade, and the declared carry "
      f"share of the {len(_pos)} cells with a positive gross runs "
      f"{_pos.median():.2f} at the median.")
print("=" * 78)
print("\nSTANDING CAVEAT")
print("-" * 78)
print("This is the fifth pass over the same CA panel. The nominal bar quoted "
      f"above is the honest one for 23 declared cells, but the STRUCTURE being "
      "tested was chosen after four prior searches over the same data, and no "
      "single-rule null bar can undo that. Block 4's verdict -- CA-vs-fly is "
      "dead as a systematic strategy -- stands, and this block adds the reason "
      "the published version of it does not rescue the idea: the entry "
      "conditions the note treats as one signal are, on SOFR, three different "
      "signals that rarely agree.")

VERDICT
* The rule AS PUBLISHED does not trade. At the note's own thresholds the five-way conjunction is satisfied on 2 of 1409 dates across GREENS/BLUES/GOLDS, and the headline cell opens twice in 4.7 years for a net of $-1,189,942.
* The reason is measurable and it is in the note's own conditions. 'Wide to the model' fights 'implied rich' (lift 0.34) and 'positioning stretched' (lift 0.43) on SOFR, and the note's own positioning mechanism runs the other way here (corr -0.187).
* Widened to 1 sigma the framework trades 4-23 times. On the ANNUALISED clock it still fails: best net Sharpe on the panel +0.2020, best on the ENGINE +0.1657, against E[max SR | null] = 0.9066 at 23 declared trials over 4.68 tradeable years, and 0 of 23 cells clear it at any cost level.
* On the PER-HOLD clock it is not nothing GROSS and is nothing NET. 8 of 23 cells clear their own per-hold bar gross -- best +1.3346 against a bar of 0.9808 on four trades -- and 0 of 23 clear it net of 1x costs. The best share